# Exploratory Data Analysis — NHANES T2DM Cohort (1999–2018)

**Purpose.** Derive the analytical cohort of adults with type 2 diabetes from the merged
NHANES 1999–2018 dataset, then characterise it: exclusion funnel, missingness, descriptive
statistics, distributions of the clustering features, and correlation structure. The output of
Section 3 (`t2d_fast_clean.csv`) is the input to the consensus clustering pipeline.

**Input.** `nhanes_1999_2018_combined_new.csv` — the combined, harmonised file produced by the
data-merging notebook (one row per participant, keyed by `SEQN`).

**Output.** `t2d_fast_clean.csv` — the final analytical cohort, plus the figures and tables
rendered inline below.

**How to run.** Run cells top to bottom from the folder containing the input CSV. Sections 4
onward depend on objects created in Section 3 (`t2d_fast_clean` and the counters `n0`–`n4`), so
cells are not independently runnable. Requires `pandas`, `numpy`, `matplotlib`, `seaborn` and
`scipy`.

---

## Cohort definition

| Step | Criterion | Rationale |
|---|---|---|
| 1 | Exclude currently pregnant participants | Pregnancy alters glucose, insulin and BMI enough to distort metabolic phenotyping |
| 2 | Restrict to age ≥ 20 | Standard NHANES adult threshold; several components are adult-only |
| 3 | Apply WHO diabetes criteria (any one sufficient) | Physician diagnosis, HbA1c ≥ 6.5%, fasting glucose ≥ 7.0 mmol/L with ≥ 8 h fast, random glucose ≥ 11.1, 2-h OGTT ≥ 11.1, insulin use, or antidiabetic medication |
| 4 | Require complete BMI, HbA1c, HOMA-IR, HOMA-β | Clustering needs complete cases; HOMA is defined only for ≥ 8 h fasting, so this also enforces the fasting restriction |

The criteria in step 3 are OR-ed rather than AND-ed because each marker is measured in a
different NHANES subsample; requiring several would select the intersection of subsamples
rather than the set of people who actually have diabetes. Missing values are safe in this
construction, since `NaN >= threshold` evaluates to `False` in pandas.

---

## Notebook contents

| Section | Content |
|---|---|
| 1 | Imports |
| 2 | Data loading |
| 3 | Exclusion pipeline & cohort derivation |
| 4 | Exclusion funnel summary |
| 5 | Missing data |
| 6 | Continuous variable descriptive statistics |
| 7 | Categorical variable frequencies |
| 8 | Age & BMI distributions |
| 9 | Clustering feature distributions (raw vs log-transformed) |
| 10 | Correlation heatmap |
| 11 | Clustering feature pairplot |
| 12 | Outlier boxplots |
| 13 | Demographic distributions |
| 14 | NHANES cycle distribution |

---

## Known issues to review before use

Spotted while documenting and **left unchanged**, since the analysis logic was preserved as-is:

1. **Type 1 diabetes is not excluded.** The step-3 criteria identify *diabetes*, not specifically
   type 2. Insulin use is itself an inclusion criterion, so insulin-treated type 1 participants
   enter the cohort. Consider whether an age-at-diagnosis rule or an insulin-without-oral-agents
   rule is needed, and state the choice in the manuscript.
2. **Case ascertainment is not uniform across cycles.** `Two_hour_glucose` exists only for the
   2005–2016 cycles, and the fasting subsample (which drives `Fasting_glucose` and the HOMA
   indices) varies by cycle. A participant's chance of meeting a given criterion therefore
   depends on which cycle they were surveyed in. The cycle distribution in Section 14 is the
   place to check how much this matters.
3. **Survey weights are not applied.** NHANES uses a complex, stratified, multistage design with
   sampling weights. Every count, percentage and correlation in this notebook is unweighted and
   describes the sample rather than the U.S. population. That is defensible for phenotyping, but
   should be stated explicitly wherever these numbers appear.



## 1 · Imports

In [ ]:
# Suppress warnings so long EDA output stays readable.
# CAUTION for reuse: this hides *all* warnings, including pandas/seaborn
# deprecation notices and numeric RuntimeWarnings (e.g. log of a non-positive
# value). Comment this out when debugging or upgrading library versions.
import warnings
warnings.filterwarnings('ignore')

import numpy as np                      # numeric arrays, log transforms
import pandas as pd                     # tabular data handling
import matplotlib.pyplot as plt         # base plotting
import matplotlib.ticker as ticker      # axis tick formatting
import seaborn as sns                   # statistical plots on top of matplotlib
from scipy.stats import gaussian_kde    # kernel density curves overlaid on histograms

# Publication-quality defaults
# Set once here so every figure below is visually consistent and export-ready,
# rather than repeating styling arguments in each plotting cell.
plt.rcParams.update({
    # Sans-serif family with fallbacks: Arial/Helvetica are typical journal
    # requirements; DejaVu Sans is the matplotlib default that always exists,
    # so figures still render if the first two are unavailable.
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    # Deliberately small type: figures are shrunk when placed in a manuscript,
    # so oversized labels become disproportionate after scaling.
    'font.size': 10, 'axes.titlesize': 11, 'axes.labelsize': 10,
    'xtick.labelsize': 9, 'ytick.labelsize': 9,
    'legend.fontsize': 9, 'figure.dpi': 120,
    # 300 dpi on save is the usual journal minimum for raster figures;
    # 'tight' bounding box trims whitespace so nothing is clipped.
    'savefig.dpi': 300, 'savefig.bbox': 'tight',
    # Drop the top/right frame lines: reduces non-data ink and is the
    # conventional look for published charts.
    'axes.spines.top': False, 'axes.spines.right': False,
})

print('Imports ready.')


## 2 · Data Loading

In [ ]:
# Load the combined 1999-2018 dataset produced by the data-merging notebook.
# Path is relative, so run this notebook from the folder holding the CSV.
df = pd.read_csv('nhanes_1999_2018_combined_new.csv')

# Print the shape and the cycles present as an immediate sanity check: if a
# cycle is missing here, the merge step upstream did not complete as expected.
print(f'Full NHANES dataset : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Cycles present      : {sorted(df["Cycle"].unique())}')
df.info()   # dtypes and non-null counts — first look at where data is missing


## 3 · Exclusion Pipeline & Cohort Derivation

Participants are filtered sequentially according to the study eligibility criteria.

In [ ]:
# ── Step 1: Exclude pregnant women ──────────────────────────────────────────
# Pregnancy alters glucose, insulin and BMI enough to distort metabolic
# phenotyping, so pregnant participants are removed before anything else.
n0 = len(df)   # record the starting N so the funnel table below can be built
# Keep participants who are NOT coded as currently pregnant (Pregnancy != 1),
# OR whose pregnancy status is missing (non-pregnant men and older women are
# never asked, so NaN here means "not applicable", not "unknown pregnancy").
# Note: `!= 1` already evaluates True for NaN in pandas, so the `.isna()` arm is
# redundant, but it is kept because it states the intent explicitly.
df_preg = df[(df['Pregnancy'] != 1) | (df['Pregnancy'].isna())].copy()
df_preg = df_preg.drop(columns=['Pregnancy'])   # column is constant now, so drop it
n1 = len(df_preg)
print(f'After excluding pregnant women      : {n1:,}  (excluded: {n0-n1:,})')

# ── Step 2: Restrict to adults aged ≥ 20 ─────────────────────────────────────
# NOTE: Age > 19 is equivalent to Age >= 20 for integer ages.
# 20 years is the standard NHANES adult threshold: several questionnaire and
# lab components (including the diabetes questionnaire) are adult-only, and
# paediatric diabetes is a different clinical entity.
df_age = df_preg[df_preg['Age'] >= 20].copy()
n2 = len(df_age)
print(f'After restricting to age ≥ 20       : {n2:,}  (excluded: {n1-n2:,})')

# ── Step 3: Identify T2DM using WHO criteria ──────────────────────────────────
# Any ONE of the criteria below is sufficient (they are OR-ed), mirroring
# clinical practice where a single diagnostic route confirms diabetes.
# Why OR rather than AND: each marker is measured in a different NHANES
# subsample, so requiring several would shrink the cohort to the intersection
# of subsamples rather than the set of people who actually have diabetes.
# Missing values are safe here: in pandas, NaN >= threshold evaluates to False,
# so an unmeasured participant is simply not captured by that arm.
t2d = df_age[
    (df_age['Diabetes'] == 1) |                          # Physician diagnosis
    (df_age['HbA1c'] >= 6.5) |                           # HbA1c ≥ 6.5%
    ((df_age['Fasting_glucose'] >= 7.0) &                # Fasting glucose ≥ 7.0 mmol/L
     (df_age['Fasting_hours'] >= 8)) |                   # (valid fasting only)
    (df_age['Glucose'] >= 11.1) |                        # Random glucose ≥ 11.1 mmol/L
    (df_age['Two_hour_glucose'] >= 11.1) |               # 2-hr OGTT ≥ 11.1 mmol/L
    (df_age['Insulin_pill'] == 1) |                      # Insulin use
    (df_age['Diabetes_pill'] == 1)                       # Antidiabetic medication
].copy()
n3 = len(t2d)
print(f'After applying WHO T2DM criteria    : {n3:,}  (excluded: {n2-n3:,})')

# ── Step 4: Require complete clustering features ──────────────────────────────
# BMI, HbA1c, HOMA_IR, HOMA_B must all be non-missing.
# HOMA values are only populated for participants with ≥ 8 hr fasting,
# so this implicitly enforces the fasting restriction as well.
# This is a complete-case restriction: clustering cannot run on partial rows,
# and it is applied last so the funnel shows its cost separately.
t2d_fast_clean = t2d.dropna(subset=['BMI', 'HbA1c', 'HOMA_IR', 'HOMA_B']).copy()
n4 = len(t2d_fast_clean)
print(f'After requiring complete features    : {n4:,}  (excluded: {n3-n4:,})')

# Persist the analytical cohort so the clustering notebooks can start from here
# rather than re-running this pipeline.
t2d_fast_clean.to_csv('t2d_fast_clean.csv', index=False)
print(f'\nFinal analytical cohort saved: t2d_fast_clean.csv')


## 4 · Exclusion Funnel Summary

In [ ]:
# Assemble the CONSORT-style exclusion funnel from the counters set in the
# previous cell. This cell therefore depends on Section 3 having been run.
# Each tuple is (label, N remaining after this step, N excluded at this step).
steps = [
    ('Full NHANES dataset (1999–2018)',             n0,    0),
    ('Exclude pregnant women',                     n1, n0-n1),
    ('Restrict to age ≥ 20 years',                 n2, n1-n2),
    ('Apply WHO T2DM diagnostic criteria',         n3, n2-n3),
    ('Require complete clustering features',       n4, n3-n4),
]

funnel = pd.DataFrame(steps, columns=['Step', 'N_remaining', 'N_excluded'])
# Retention relative to the ORIGINAL dataset (not the previous step), which is
# the figure normally reported in a flow diagram.
funnel['% of total'] = (funnel['N_remaining'] / n0 * 100).round(1)
print(funnel.to_string(index=False))   # to_string avoids truncating the labels


## 5 · Missing Data

In [ ]:
# ── Tabular summary ──────────────────────────────────────────────────────────
# Missingness is assessed on the FINAL cohort, so the four clustering features
# (BMI, HbA1c, HOMA_IR, HOMA_B) are complete by construction and will not
# appear here. What remains tells us which covariates are usable downstream.
missing       = t2d_fast_clean.isnull().sum().sort_values(ascending=False)
missing_pct   = (missing / len(t2d_fast_clean) * 100).round(1)
missing_df    = pd.DataFrame({'Missing_n': missing, 'Missing_%': missing_pct})
missing_df    = missing_df[missing_df['Missing_n'] > 0]   # keep only affected variables
print('Variables with missing values:')
print(missing_df.to_string())

# ── Missing data bar chart ────────────────────────────────────────────────────
# Guarded so the cell still runs cleanly on a cohort with no missing values.
if len(missing_df) > 0:
    fig, ax = plt.subplots(figsize=(10, 5))
    # Horizontal bars keep long variable names legible without rotation.
    missing_df['Missing_%'].plot.barh(ax=ax, color='#d62728', edgecolor='white')
    ax.set_xlabel('Missing (%)')
    ax.set_title('Missing Data by Variable — Analytical Cohort', fontweight='bold')
    # Reference lines at conventional decision points: below ~5% missingness is
    # usually negligible, above ~20% a variable is often dropped or imputed
    # with caution rather than used directly.
    ax.axvline(5,  color='orange', ls='--', lw=0.9, label='5%')
    ax.axvline(20, color='red',    ls='--', lw=0.9, label='20%')
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print('No missing values in analytical cohort.')


## 6 · Continuous Variable Descriptive Statistics

In [ ]:
# Continuous variables to summarise. This list is deliberately broader than the
# clustering features: it also covers the covariates used later for adjustment
# and for describing the cohort in the manuscript's Table 1.
continuous_vars = [
    'Age', 'Family_PIR', 'BMI', 'Systolic_BP', 'Diastolic_BP',
    'HbA1c', 'LDL', 'HDL', 'Triglycerides',
    'Fasting_glucose', 'Fasting_insulin', 'HOMA_IR', 'HOMA_B',
    'Creatinine', 'eGFR', 'Fasting_hours',
    'Age_at_diagnosis', 'Diabetes_duration'
]

# FIX: was incorrectly referencing `df` for Q1/Q3 — now uses t2d_fast_clean
# Transpose (.T) puts one variable per ROW, which reads better than one per
# column when there are ~18 variables.
desc = t2d_fast_clean[continuous_vars].agg(
    ['count', 'mean', 'std', 'median', 'min', 'max']
).T
# Quartiles are added separately because .agg() does not accept a quantile
# argument; the Series index aligns on variable name, so assignment is safe.
desc['Q1']  = t2d_fast_clean[continuous_vars].quantile(0.25)
desc['Q3']  = t2d_fast_clean[continuous_vars].quantile(0.75)
# IQR is reported alongside the mean/SD because several of these variables
# (HOMA indices, triglycerides) are right-skewed, where median/IQR is the
# more faithful summary.
desc['IQR'] = desc['Q3'] - desc['Q1']

print('Descriptive statistics — analytical cohort (t2d_fast_clean):')
print(desc.round(2).to_string())


## 7 · Categorical Variable Frequencies

In [ ]:
categorical_vars = [
    'Sex', 'Ethnicity', 'Education_level',
    'Smoking_category', 'Alcohol_status', 'Physical_activity', 'Cycle'
    # NOTE: 'Diabetes' excluded — constant after T2DM filter
]

# One frequency table per variable. dropna=False is important: it counts
# missing as its own category, so the reader can see non-response rather than
# having it silently vanish from the denominator.
for col in categorical_vars:
    print(f'\n{"="*40}')
    print(f'  {col}')
    print(f'{"="*40}')
    counts = t2d_fast_clean[col].value_counts(dropna=False)
    # Percentages use the full cohort size as the denominator, so the column
    # sums to 100% including any missing category.
    pct    = (counts / len(t2d_fast_clean) * 100).round(1)
    print(pd.DataFrame({'Count': counts, 'Percent (%)': pct}).to_string())


## 8 · Age & BMI Distributions

In [ ]:
# Two panels side by side so age and BMI can be compared at a glance.
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Age distribution
# kde=True overlays a smoothed density on the histogram, which makes the shape
# (unimodal? skewed?) easier to judge than bar heights alone.
sns.histplot(t2d_fast_clean['Age'], bins=30, kde=True,
             color='#4292c6', edgecolor='white', ax=axes[0])
# Median rather than mean is marked because it is robust to the skew and the
# top-coding NHANES applies to old ages (ages above the cap are collapsed).
axes[0].axvline(t2d_fast_clean['Age'].median(), color='darkblue',
                ls='--', lw=1.5, label=f'Median = {t2d_fast_clean["Age"].median():.1f}')
axes[0].set_xlabel('Age (years)')
axes[0].set_title('Age Distribution', fontweight='bold')
axes[0].legend()

# BMI distribution
sns.histplot(t2d_fast_clean['BMI'], bins=30, kde=True,
             color='#41ab5d', edgecolor='white', ax=axes[1])
# BMI is right-skewed (a long tail of high values), so the median again gives
# a better sense of the typical participant than the mean would.
axes[1].axvline(t2d_fast_clean['BMI'].median(), color='darkgreen',
                ls='--', lw=1.5, label=f'Median = {t2d_fast_clean["BMI"].median():.1f}')
axes[1].set_xlabel('BMI (kg/m²)')
axes[1].set_title('BMI Distribution', fontweight='bold')
axes[1].legend()

fig.suptitle('Age & BMI — Analytical Cohort', fontsize=12, fontweight='bold')
plt.tight_layout()   # prevents the suptitle and axis labels from overlapping
plt.show()


## 9 · Clustering Feature Distributions

Raw distributions are shown first to confirm right skew in HOMA indices, followed by log-transformed versions which are used in the clustering pipeline.

In [ ]:
# 2 rows x 3 columns: the top row shows each feature as measured, the bottom
# row shows the same feature after log transformation. Presenting them together
# is what justifies the transformation used in the clustering pipeline.
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# Work on a copy so the log columns never leak into the saved cohort.
df_plot = t2d_fast_clean.copy()
# HOMA-IR and HOMA-β are ratios bounded below by zero with long right tails.
# Logging them pulls the tail in, giving roughly symmetric distributions that
# distance-based clustering can handle without a few extreme cases dominating.
df_plot['log_HOMA_IR'] = np.log(df_plot['HOMA_IR'])
df_plot['log_HOMA_B']  = np.log(df_plot['HOMA_B'])

# Parallel lists: column i of the grid is described by element i of each list.
# HbA1c appears in both rows unchanged — it is already near-symmetric and is
# NOT log-transformed, so the bottom-left panel is a deliberate control.
raw_vars    = ['HbA1c',       'HOMA_IR',         'HOMA_B']
log_vars    = ['HbA1c',       'log_HOMA_IR',      'log_HOMA_B']
raw_titles  = ['HbA1c (%)',   'HOMA-IR (raw)',    'HOMA-β (raw)']
log_titles  = ['HbA1c (%)',   'log(HOMA-IR)',     'log(HOMA-β)']
colors      = ['#6baed6',     '#74c476',           '#fdae6b']

# Outer loop walks the three columns (one metabolic feature each).
for col_i, (raw_v, log_v, raw_t, log_t, clr) in enumerate(
        zip(raw_vars, log_vars, raw_titles, log_titles, colors)):

    # Inner loop walks the two rows: row 0 = raw scale, row 1 = log scale.
    for row_i, (var, ttl) in enumerate([(raw_v, raw_t), (log_v, log_t)]):
        ax   = axes[row_i][col_i]
        data = df_plot[var].dropna()   # gaussian_kde cannot accept NaN

        ax.hist(data, bins=35, color=clr, edgecolor='white', alpha=0.8)

        # Overlay a kernel density estimate on the same axis as the histogram.
        kde     = gaussian_kde(data)
        x_vals  = np.linspace(data.min(), data.max(), 500)   # smooth evaluation grid
        # gaussian_kde returns a probability density (area = 1), but the
        # histogram is in raw counts. Multiplying by n and by the bin width
        # (range / number of bins) rescales the curve onto the count axis so
        # the two are directly comparable.
        kde_sc  = kde(x_vals) * len(data) * (data.max() - data.min()) / 35
        ax.plot(x_vals, kde_sc, color=clr, lw=2)

        # Annotate mean and SD: on the log panels these describe the
        # transformed scale, which is the scale clustering actually sees.
        m, s = data.mean(), data.std()
        ax.axvline(m, color='black', ls='--', lw=1.2,
                   label=f'Mean={m:.2f}  SD={s:.2f}')
        ax.set_title(f'{ttl}  ({"raw" if row_i==0 else "log-transformed"})',
                     fontweight='bold')
        ax.legend(fontsize=7.5)
        ax.set_ylabel('Frequency')

fig.suptitle('Clustering Feature Distributions — Raw vs Log-Transformed',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


## 10 · Correlation Heatmap — Continuous Variables

In [ ]:
# Continuous variables for the correlation matrix: the five clustering
# features plus the metabolic and renal markers most likely to be collinear
# with them.
corr_vars = [
    'Age', 'BMI', 'HbA1c', 'HOMA_IR', 'HOMA_B',
    'Fasting_glucose', 'Fasting_insulin', 'HDL', 'LDL',
    'Triglycerides', 'Systolic_BP', 'Diastolic_BP', 'eGFR'
]

# Spearman (rank-based) rather than Pearson: it captures monotonic
# relationships without assuming linearity or normality, and is insensitive to
# the skew and outliers present in the HOMA and lipid variables. It also means
# the result is unchanged by the log transformation applied elsewhere.
corr = t2d_fast_clean[corr_vars].corr(method='spearman')

fig, ax = plt.subplots(figsize=(11, 9))
# Mask the upper triangle: the matrix is symmetric, so showing both halves
# doubles the ink without adding information.
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
    # Fix the scale to the full [-1, 1] range so the colour of a cell means the
    # same thing here as in any other correlation figure, and so the diverging
    # palette is centred on zero.
    vmin=-1, vmax=1, linewidths=0.4, linecolor='white',
    annot_kws={'size': 7.5}, ax=ax
)
ax.set_title('Spearman Correlation Matrix — Analytical Cohort',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


## 11 · Clustering Feature Pairplot

In [ ]:
# Pairwise view of the five clustering features, to check for separation,
# nonlinearity or clumping before any clustering algorithm is applied.
df_pair = t2d_fast_clean[['Age', 'BMI', 'HbA1c', 'HOMA_IR', 'HOMA_B']].copy()
# Use the log scale for the HOMA indices, matching what the clustering
# pipeline consumes, so this plot reflects the actual feature space.
df_pair['log_HOMA_IR'] = np.log(df_pair['HOMA_IR'])
df_pair['log_HOMA_B']  = np.log(df_pair['HOMA_B'])
df_pair = df_pair.drop(columns=['HOMA_IR', 'HOMA_B'])   # drop raw versions to avoid duplication

g = sns.pairplot(
    # A full pairplot draws every point in every panel, which is slow and
    # produces an unreadable solid mass. A fixed random_state keeps the
    # subsample reproducible; min() guards against a cohort smaller than 2000.
    df_pair.sample(min(2000, len(df_pair)), random_state=42),  # subsample for speed
    diag_kind='kde',                                            # densities on the diagonal
    # Low alpha reveals density in overlapping regions; rasterized keeps the
    # exported vector file small; s=8 shrinks markers to reduce overplotting.
    plot_kws={'alpha': 0.25, 'rasterized': True, 's': 8},
    diag_kws={'fill': True},
    corner=True                                                 # lower triangle only (matrix is symmetric)
)
g.figure.suptitle('Pairplot of Clustering Features (n=2,000 subsample)',
                   y=1.01, fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


## 12 · Outlier Boxplots — Clustering Features

In [ ]:
# Boxplots to quantify outliers in the clustering features before deciding
# whether any trimming or winsorising is warranted.
df_box = t2d_fast_clean[['Age', 'BMI', 'HbA1c']].copy()
# HOMA indices are shown on the log scale: on the raw scale nearly every high
# value is flagged as an outlier purely because of the skew, which would
# overstate the problem.
df_box['log_HOMA_IR'] = np.log(t2d_fast_clean['HOMA_IR'])
df_box['log_HOMA_B']  = np.log(t2d_fast_clean['HOMA_B'])

fig, axes = plt.subplots(1, 5, figsize=(16, 5))   # one panel per feature
colors_box = ['#4292c6', '#41ab5d', '#e6550d', '#756bb1', '#fd8d3c']

for ax, col, clr in zip(axes, df_box.columns, colors_box):
    bp = ax.boxplot(
        df_box[col].dropna(), patch_artist=True, notch=True,
        # notch=True draws a confidence interval around the median: if the
        # notches of two boxes did not overlap, their medians would differ.
        medianprops={'color': 'black', 'lw': 2},
        boxprops={'facecolor': clr, 'alpha': 0.7},
        # Small, faint flier markers so hundreds of outliers do not swamp the box.
        flierprops={'marker': 'o', 'markersize': 2, 'alpha': 0.3, 'markerfacecolor': clr}
    )
    # Recompute the outlier count explicitly with Tukey's rule so the number can
    # be printed in the title; matplotlib draws the fliers but does not expose
    # a count directly.
    q1 = df_box[col].quantile(0.25)
    q3 = df_box[col].quantile(0.75)
    iqr = q3 - q1
    n_out = ((df_box[col] < q1 - 1.5*iqr) | (df_box[col] > q3 + 1.5*iqr)).sum()
    ax.set_title(f'{col}\n({n_out} outliers)', fontsize=9, fontweight='bold')
    ax.set_xticks([])   # a single box per panel needs no category tick

fig.suptitle('Boxplots — Clustering Features', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


## 13 · Demographic & Lifestyle Distributions

All plots use `t2d_fast_clean` (the final analytical cohort).

In [ ]:
# FIX: was using `t2d` (pre-filter) — now correctly uses `t2d_fast_clean`
plot_vars = [
    'Sex', 'Ethnicity', 'Education_level',
    'Smoking_category', 'Alcohol_status', 'Physical_activity'
]

# NHANES codes sex numerically (RIAGENDR: 1 = male, 2 = female). Map to labels
# so the chart is readable without consulting the codebook.
sex_labels = {1.0: 'Male', 2.0: 'Female'}
t2d_fast_clean['Sex_label'] = t2d_fast_clean['Sex'].map(sex_labels)

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()   # flatten the 2x3 grid so it can be indexed with a single counter

for i, var in enumerate(plot_vars):
    ax      = axes[i]
    # Substitute the labelled column for Sex; every other variable is already text.
    col     = 'Sex_label' if var == 'Sex' else var
    # Order bars by frequency (descending) so the dominant categories read first.
    order   = t2d_fast_clean[col].value_counts().index
    sns.countplot(x=col, data=t2d_fast_clean, order=order,
                  palette='Set2', ax=ax, edgecolor='white')
    ax.set_title(f'{var}', fontweight='bold')
    ax.tick_params(axis='x', rotation=35)   # rotate so long category names do not collide
    ax.set_xlabel('')                        # the title already names the variable
    # Denominator excludes missing values, so percentages describe the
    # distribution among respondents for that variable. This means the
    # denominator can differ between panels.
    total = t2d_fast_clean[col].notna().sum()
    for p in ax.patches:
        h = p.get_height()
        if h > 0:                            # skip empty categories
            # Annotate each bar with its share, since raw counts alone make
            # panels with different denominators hard to compare.
            ax.annotate(f'{100*h/total:.1f}%',
                        (p.get_x() + p.get_width()/2, h),
                        ha='center', va='bottom', fontsize=8.5)

fig.suptitle('Demographic & Lifestyle Distributions — Analytical Cohort',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# Clean up helper column
# Removing Sex_label keeps the in-memory cohort identical to the saved CSV, so
# later cells cannot accidentally depend on a plotting-only column.
t2d_fast_clean = t2d_fast_clean.drop(columns=['Sex_label'], errors='ignore')


## 14 · NHANES Cycle Distribution

In [ ]:
# How the cohort is distributed across survey cycles. Uneven representation
# matters because NHANES protocols changed over time, so a cycle imbalance can
# translate into a measurement-method imbalance.
cycle_counts = t2d_fast_clean['Cycle'].value_counts().sort_index()
cycle_df     = cycle_counts.reset_index()
cycle_df.columns = ['Cycle', 'Count']

# Extract start year and compute end year (e.g., 1999_2000 → end=2000)
# The cycle label is a string like "1999_2000"; splitting on the underscore
# yields numeric years that can be placed on a real time axis.
cycle_df['Start_year'] = cycle_df['Cycle'].str.split('_').str[0].astype(int)
cycle_df['End_year']   = cycle_df['Cycle'].str.split('_').str[1].astype(int)
cycle_df = cycle_df.sort_values('End_year')   # chronological order for the line chart

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Line chart: x-axis at 2-year intervals 2000, 2002, …, 2018 ───────────────
# Plotting against the end year spaces the points at true two-year intervals,
# so any trend in cohort size is not distorted by evenly spaced categories.
axes[0].plot(cycle_df['End_year'], cycle_df['Count'],
             marker='o', ls='-', color='teal', lw=2, zorder=3)   # zorder=3 keeps the line above the grid
for _, row in cycle_df.iterrows():
    # Print each count just above its marker so exact Ns are readable.
    axes[0].text(row['End_year'], row['Count'] + 8,
                 str(int(row['Count'])), ha='center', fontsize=8.5)

tick_years = list(range(2000, 2019, 2))   # 2000, 2002, 2004, …, 2018
axes[0].set_xticks(tick_years)
axes[0].set_xticklabels([str(y) for y in tick_years], rotation=45, ha='right')
axes[0].set_xlim(1998, 2020)   # padding so the first and last markers are not clipped
axes[0].set_xlabel('NHANES Cycle (end year)', labelpad=8)
axes[0].set_ylabel('N participants with T2DM')
axes[0].set_title('NHANES Cycle Participation Trend', fontweight='bold')
axes[0].grid(True, ls='--', alpha=0.5)

# ── Pie chart ─────────────────────────────────────────────────────────────────
# Same data as a share of the total: makes it obvious whether any single cycle
# contributes a disproportionate fraction of the cohort.
axes[1].pie(
    cycle_df['Count'], labels=cycle_df['Cycle'],
    autopct='%1.1f%%', startangle=140,
    colors=plt.cm.Set3.colors, wedgeprops={'edgecolor': 'white'}
)
axes[1].set_title('Cycle Proportions', fontweight='bold')

fig.suptitle('NHANES Cycle Distribution — Analytical Cohort',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()
